### Distillation

In [ ]:
import subprocess
import os
from tqdm.notebook import tqdm

In [ ]:
MODELS = ["dinov2_vitb"]
DATASET = "SIPFARE10"
DATASET_PATH = f"/lustre/fswork/projects/rech/rbw/ucw75ke/datasets/{DATASET}"
results = {m: {} for m in MODELS}
DISTILL_CONFIGS = {
    "dinov2_vitb": [{"syn_res": 140, "real_res": 140, "crop_res": 140, "train_crop_mode": "random", "augs_per_batch": 3, "ipc": 1}],
}

In [ ]:


# Aplatir tous les runs à faire
all_runs = [
    (model, cfg)
    for model, configs in DISTILL_CONFIGS.items()
    for cfg in configs
]

for model, config in tqdm(all_runs, desc="Distillation runs", unit="run"):
    run_name = f"sipfar_{model}_distill_{config['syn_res']}_ipc{config['ipc']}_augs{config['augs_per_batch']}"
    
    tqdm.write(f"\n{'='*50}")
    tqdm.write(f"Distilling {model} | ipc={config['ipc']} | syn_res={config['syn_res']} | augs={config['augs_per_batch']}")
    tqdm.write(f"Run name: {run_name}")
    tqdm.write('='*50)

    env = os.environ.copy()
    env["DATASET"] = DATASET
    env["MODEL"] = model

    process = subprocess.Popen(
        [
            "./run.sh", "distill",
            f"--augs_per_batch={config['augs_per_batch']}",
            f"--syn_res={config['syn_res']}",
            f"--real_res={config['real_res']}",
            f"--crop_res={config['crop_res']}",
            f"--train_crop_mode={config['train_crop_mode']}",
            f"--ipc={config['ipc']}",
            f"--run_name={run_name}",
            f"--data_root={DATASET_PATH}"
        ],
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        cwd="/lustre/fswork/projects/rech/rbw/ucw75ke/projects/GradientDistillation"
    )

    for line in process.stdout:
        tqdm.write(line, end="")

    process.wait()
    tqdm.write(f"\nReturn code: {process.returncode}")

### Visualization

In [ ]:
import torch
import matplotlib.pyplot as plt
import torchvision.utils as vutils

# Load
run_name = "sipfar_dinov2_vitb_distill_140_ipc1_augs3_s3407"
data = torch.load(f"/lustre/fswork/projects/rech/rbw/ucw75ke/projects/GradientDistillation/logged_files/distillation/sipfar/dinov2_vitb//{run_name}/data.pth", weights_only=False)

images = data["images"]  # (N, C, H, W)
print(f"Shape: {images.shape}")

# Display grid
grid = vutils.make_grid(images, nrow=1, padding=2)
plt.figure(figsize=(2, 2))
plt.imshow(grid.permute(1, 2, 0).cpu().numpy())
plt.axis("off")
plt.title("Distilled Images")
plt.show()